# TP5 — Fashion-MNIST MLP versus CNN

Run all. Fashion-MNIST downloads automatically. Training uses five epochs; Colab GPU is optional. Keep the generated mlp_model.keras and cnn_model.keras for TP6.

This notebook contains the full Python solution. Board projects also require the included firmware. Read EXPLICATION_AR.md for the detailed Arabic explanation.

[Assignment](https://github.com/ahmed3991/AIoT-2025/tree/main/TP5)

In [1]:
# Install missing dependencies before importing the model libraries.
import importlib.util
import importlib.metadata
import subprocess
import sys

required = {'tensorflow': 'tensorflow', 'numpy': 'numpy', 'pandas': 'pandas', 'matplotlib': 'matplotlib'}
missing = []
for module_name, package_spec in required.items():
    try:
        found = importlib.util.find_spec(module_name) is not None
    except ModuleNotFoundError:
        found = False
    if not found:
        missing.append(package_spec)
    elif '==' in package_spec:
        distribution, expected = package_spec.split('==')
        if importlib.metadata.version(distribution) != expected:
            missing.append(package_spec)
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
print('Dependencies ready.')


Dependencies ready.


## 1. Imports and settings

In [2]:
"""TP5: train the two architectures requested in the Fashion-MNIST TP."""
import json
import os
from pathlib import Path
from time import perf_counter

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

ROOT = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
OUT = ROOT / "results"
OUT.mkdir(parents=True, exist_ok=True)
keras.utils.set_random_seed(42)
EPOCHS = 5
BATCH_SIZE = 64



## 2. Load and normalize the complete Fashion-MNIST dataset

In [3]:
(train_images, train_labels), (test_images, test_labels) = keras.datasets.fashion_mnist.load_data()
x_train_mlp = train_images.astype("float32") / 255.0
x_test_mlp = test_images.astype("float32") / 255.0
x_train_cnn = x_train_mlp[..., np.newaxis]
x_test_cnn = x_test_mlp[..., np.newaxis]
print("MLP:", x_train_mlp.shape, x_test_mlp.shape)
print("CNN:", x_train_cnn.shape, x_test_cnn.shape)



MLP: (60000, 28, 28) (10000, 28, 28)
CNN: (60000, 28, 28, 1) (10000, 28, 28, 1)


## 3. Build the exact MLP and CNN architectures from the assignment

In [4]:
mlp_model = keras.Sequential([
    layers.Input(shape=(28, 28)),
    layers.Flatten(),
    layers.Dense(256, activation="relu"),
    layers.Dense(128, activation="relu"),
    layers.Dense(10, activation="softmax"),
], name="mlp_model")

cnn_model = keras.Sequential([
    layers.Input(shape=(28, 28, 1)),
    layers.Conv2D(16, 3, activation="relu"),
    layers.MaxPooling2D(2),
    layers.Conv2D(32, 3, activation="relu"),
    layers.MaxPooling2D(2),
    layers.Flatten(),
    layers.Dense(64, activation="relu"),
    layers.Dense(10, activation="softmax"),
], name="cnn_model")

for model in [mlp_model, cnn_model]:
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    model.summary()



Model: "mlp_model"
┌─────────────────────────────────┬────────────────────────┬───────────────┐
│ Layer (type)                    │ Output Shape           │       Param # │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       200,960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘
 Total params: 235,146 (918.54 KB)
 Trainable params: 235,146 (918.54 KB)
 Non-trainable params: 0 (0.00 B)
Model: "cnn_model"
┌──────

## 4. Train five epochs and evaluate on the independent test set

In [5]:
models = {"MLP": mlp_model, "CNN": cnn_model}
histories = {}
rows = []
for name, model in models.items():
    train_x = x_train_mlp if name == "MLP" else x_train_cnn
    test_x = x_test_mlp if name == "MLP" else x_test_cnn
    start = perf_counter()
    history = model.fit(
        train_x, train_labels, epochs=EPOCHS, batch_size=BATCH_SIZE,
        validation_split=0.1, verbose=2,
    )
    training_s = perf_counter() - start
    loss, accuracy = model.evaluate(test_x, test_labels, verbose=0)
    histories[name] = history.history
    model.save(OUT / f"{name.lower()}_model.keras")
    model.save(OUT / f"{name.lower()}_model.h5", include_optimizer=False)

    # Analytical estimates: count a multiply plus an addition as two FLOPs.
    # Ignore activation, pooling, bias, and softmax operations in this estimate.
    macs = 0
    activations = 0
    for layer in model.layers:
        shape = tuple(int(d) for d in layer.output.shape[1:])
        activations += int(np.prod(shape))
        if isinstance(layer, layers.Dense):
            macs += int(np.prod(layer.kernel.shape))
        elif isinstance(layer, layers.Conv2D):
            macs += int(np.prod(shape)) * int(np.prod(layer.kernel.shape[:3]))
    parameters = int(sum(np.prod(weight.shape) for weight in model.trainable_weights))
    inference_flops = 2 * macs
    # Approximately one forward and two backward passes per batch.
    training_step_flops = 3 * inference_flops * BATCH_SIZE
    # Float32 parameters + gradients + Adam's two moment arrays = 16 bytes/P.
    core_bytes = 16 * parameters
    # Rough activation/gradient storage; excludes allocator and runtime overhead.
    estimated_training_bytes = core_bytes + 8 * BATCH_SIZE * activations
    rows.append({
        "Model": name, "Test_accuracy": accuracy, "Test_loss": loss,
        "Trainable_parameters": parameters,
        "H5_size_MB": (OUT / f"{name.lower()}_model.h5").stat().st_size / 1024**2,
        "Inference_FLOPs_estimate": inference_flops,
        "Training_step_FLOPs_estimate_batch64": training_step_flops,
        "Parameters_gradients_Adam_MB": core_bytes / 1024**2,
        "Training_memory_estimate_MB": estimated_training_bytes / 1024**2,
        "Training_s": training_s,
    })
results = pd.DataFrame(rows)
results.to_csv(OUT / "results.csv", index=False)
(OUT / "history.json").write_text(json.dumps(histories, indent=2), encoding="utf-8")
print(results.to_string(index=False))



Epoch 1/5
844/844 - 4s - 5ms/step - accuracy: 0.8228 - loss: 0.4945 - val_accuracy: 0.8482 - val_loss: 0.4070
Epoch 2/5
844/844 - 3s - 4ms/step - accuracy: 0.8660 - loss: 0.3668 - val_accuracy: 0.8542 - val_loss: 0.3942
Epoch 3/5
844/844 - 5s - 6ms/step - accuracy: 0.8797 - loss: 0.3260 - val_accuracy: 0.8653 - val_loss: 0.3657
Epoch 4/5
844/844 - 5s - 6ms/step - accuracy: 0.8897 - loss: 0.2989 - val_accuracy: 0.8687 - val_loss: 0.3620
Epoch 5/5
844/844 - 5s - 6ms/step - accuracy: 0.8966 - loss: 0.2791 - val_accuracy: 0.8750 - val_loss: 0.3464
Epoch 1/5
844/844 - 7s - 8ms/step - accuracy: 0.7979 - loss: 0.5654 - val_accuracy: 0.8440 - val_loss: 0.4443
Epoch 2/5
844/844 - 10s - 12ms/step - accuracy: 0.8624 - loss: 0.3796 - val_accuracy: 0.8648 - val_loss: 0.3903
Epoch 3/5
844/844 - 10s - 12ms/step - accuracy: 0.8811 - loss: 0.3301 - val_accuracy: 0.8730 - val_loss: 0.3641
Epoch 4/5
844/844 - 11s - 13ms/step - accuracy: 0.8910 - loss: 0.3002 - val_accuracy: 0.8798 - val_loss: 0.3440
Epoc

## 5. Plot learning curves and write the comparison report

In [6]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for name, history in histories.items():
    epochs = range(1, EPOCHS + 1)
    axes[0].plot(epochs, history["val_accuracy"], marker="o", label=name)
    axes[1].plot(epochs, history["val_loss"], marker="o", label=name)
axes[0].set(title="Validation accuracy", xlabel="Epoch", ylabel="Accuracy")
axes[1].set(title="Validation loss", xlabel="Epoch", ylabel="Loss")
for axis in axes:
    axis.legend()
    axis.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(OUT / "learning_curves.png", dpi=140)
plt.close(fig)

table = "| Model | Test accuracy | Loss | Parameters | H5 MB | Inference FLOPs* | Training step FLOPs* (batch 64) | Training memory MB* |\n|---|---|---|---|---|---|---|---|\n"
for row in rows:
    table += f"| {row['Model']} | {row['Test_accuracy']:.4f} | {row['Test_loss']:.4f} | {row['Trainable_parameters']} | {row['H5_size_MB']:.3f} | {row['Inference_FLOPs_estimate']} | {row['Training_step_FLOPs_estimate_batch64']} | {row['Training_memory_estimate_MB']:.3f} |\n"
winner = max(rows, key=lambda row: row["Test_accuracy"])["Model"]
smaller = min(rows, key=lambda row: row["Trainable_parameters"])["Model"]
report = f"""# TP5 report

Both assignment architectures were trained for five epochs, batch size 64,
Adam, sparse categorical cross-entropy. Fashion-MNIST provides 60,000 training
and 10,000 test images. A 10% validation partition is taken from the training
set; the test set is evaluated only after training. Pixels are divided by 255.

{table}
*FLOPs and training memory are analytical estimates, not hardware measurements.
One multiply-add counts as two FLOPs. Forward FLOPs include Dense and Conv2D
kernels; activations, bias, pooling, and softmax are excluded. A training step
means **one batch of 64**, approximated as three forward-pass costs per sample.
Training memory estimates parameters, gradients, two Adam moments (16 bytes
per parameter), plus approximate batch activation/gradient storage. TensorFlow
runtime, data storage, workspaces, and allocator overhead are excluded, so this
is not a measured peak. Model-file MB means bytes / 1024².

**{winner}** achieved higher test accuracy in this run; **{smaller}** has fewer
parameters. CNN filters share weights across image locations and preserve
spatial structure, helping learn local patterns. The MLP is simple and uses
fewer arithmetic operations here, but its first dense layer connects every
pixel to every hidden neuron. Fewer parameters do not necessarily mean fewer
FLOPs: convolutions reuse kernels at many locations.

![Validation learning curves](learning_curves.png)

Saved .keras files are the reusable models for TP6; .h5 files satisfy the size
comparison. Measurements are from this computer; Colab runs may differ.

Source: https://github.com/ahmed3991/AIoT-2025/blob/main/TP5/TP5.md
"""
(OUT / "REPORT.md").write_text(report, encoding="utf-8")
print("Saved trained models, curves, metrics, and REPORT.md in", OUT)


Saved trained models, curves, metrics, and REPORT.md in C:\Users\chikh\Desktop\UNIV\ghanabzia\TP5-Ghanabzia\results


In [7]:
from IPython.display import display, Markdown
display(Markdown((OUT).joinpath('REPORT.md').read_text(encoding='utf-8')))


# TP5 report

Both assignment architectures were trained for five epochs, batch size 64,
Adam, sparse categorical cross-entropy. Fashion-MNIST provides 60,000 training
and 10,000 test images. A 10% validation partition is taken from the training
set; the test set is evaluated only after training. Pixels are divided by 255.

| Model | Test accuracy | Loss | Parameters | H5 MB | Inference FLOPs* | Training step FLOPs* (batch 64) | Training memory MB* |
|---|---|---|---|---|---|---|---|
| MLP | 0.8693 | 0.3745 | 235146 | 0.916 | 469504 | 90144768 | 4.163 |
| CNN | 0.8807 | 0.3452 | 56714 | 0.245 | 1413504 | 271392768 | 10.175 |

*FLOPs and training memory are analytical estimates, not hardware measurements.
One multiply-add counts as two FLOPs. Forward FLOPs include Dense and Conv2D
kernels; activations, bias, pooling, and softmax are excluded. A training step
means **one batch of 64**, approximated as three forward-pass costs per sample.
Training memory estimates parameters, gradients, two Adam moments (16 bytes
per parameter), plus approximate batch activation/gradient storage. TensorFlow
runtime, data storage, workspaces, and allocator overhead are excluded, so this
is not a measured peak. Model-file MB means bytes / 1024².

**CNN** achieved higher test accuracy in this run; **CNN** has fewer
parameters. CNN filters share weights across image locations and preserve
spatial structure, helping learn local patterns. The MLP is simple and uses
fewer arithmetic operations here, but its first dense layer connects every
pixel to every hidden neuron. Fewer parameters do not necessarily mean fewer
FLOPs: convolutions reuse kernels at many locations.

![Validation learning curves](learning_curves.png)

Saved .keras files are the reusable models for TP6; .h5 files satisfy the size
comparison. Measurements are from this computer; Colab runs may differ.

Source: https://github.com/ahmed3991/AIoT-2025/blob/main/TP5/TP5.md
